In [ ]:
from tradepy.target import create_targets, create_targets_atr, apply_liquidity_filter, audit_single_df
from tradepy.models import prepare_all
import numpy as np
import torch.nn as nn
import torch
from tqdm import tqdm
from torch import amp
from sklearn.preprocessing import RobustScaler
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd


In [2]:
asset_list = ["bp1", "cl1", "es1", "gc1"]
ASSET = asset_list[3]
MINUTES = 60
RETURN_HORIZON_MIN  = MINUTES * 2

In [3]:
df_final, spec = prepare_all(ASSET, MINUTES, RETURN_HORIZON_MIN, json_config_path="../Features/features_config.json")

Dataset 'gc1' imported with 4565856 rows
Limpieza gc1 completada. Filas restantes: 4456388


In [4]:
# Columnas a excluir siempre
exclude = [
    # Identificadores / temporales no cíclicos
    "datetime", "dtyyyymmdd", "trading_date", "ticker", "per", "openint",

    # OHLCV raw (el modelo no debería ver precios absolutos)
    "open", "high", "low", "close", "volume",

    # Targets
    "target_bin", "target_class",
]

feature_cols = [c for c in df_final.columns if c not in exclude]
df_feats = df_final[feature_cols].copy()

In [6]:
df_final_bin = create_targets(df_final, return_horizon_min=RETURN_HORIZON_MIN, sampling_minutes=MINUTES, tick_size=spec['tick_size'], threshold_buy=20)
print(df_final_bin.target_bin.sum() / df_final_bin['datetime'].dt.date.nunique())

4.1623762376237625


In [8]:
df_final_triple = create_targets(df_final, return_horizon_min=RETURN_HORIZON_MIN, sampling_minutes=MINUTES, tick_size=spec['tick_size'], threshold_buy=25 , threshold_sell=25, ternary=True)
print(len(df_final_triple[df_final_triple.target_class == 2]) / df_final_triple['datetime'].dt.date.nunique())
print(len(df_final_triple[df_final_triple.target_class == 0]) / df_final_triple['datetime'].dt.date.nunique())

3.3804455445544552
3.2935643564356436


In [9]:
df_final_bin_atr = create_targets_atr(df_final, return_horizon_min=RETURN_HORIZON_MIN, sampling_minutes=MINUTES, tick_size=spec['tick_size'], ternary=False, atr_multiplier=1.5)
print(df_final_bin_atr.target_bin.sum() / df_final_bin_atr['datetime'].dt.date.nunique())

1.0133663366336634


In [10]:
df_final_triple_atr = create_targets_atr(df_final, return_horizon_min=RETURN_HORIZON_MIN, sampling_minutes=MINUTES, tick_size=spec['tick_size'], ternary=True, atr_multiplier=1.5)
print(len(df_final_triple_atr[df_final_triple_atr.target_class == 2]) / df_final_triple_atr['datetime'].dt.date.nunique())
print(len(df_final_triple_atr[df_final_triple_atr.target_class == 0]) / df_final_triple_atr['datetime'].dt.date.nunique())

1.0133663366336634
1.0155940594059405


In [11]:
df_final_triple_liquid = apply_liquidity_filter(df_final_triple_atr)

In [12]:
print(len(df_final_triple_liquid[df_final_triple_liquid.target_class == 2]) / df_final_triple_liquid['datetime'].dt.date.nunique())
print(len(df_final_triple_liquid[df_final_triple_liquid.target_class == 0]) / df_final_triple_liquid['datetime'].dt.date.nunique())

0.6824257425742575
0.6589108910891089


In [16]:
df_final_triple_liquid = audit_single_df(df_final_triple_liquid, feature_cols)

💎 Sin NaNs en features


In [19]:
TARGET_COLUMN_BIN = 'target_bin' # 1 si sube, 0 si baja/lateral
TARGET_COLUMN_TRIPLE = 'target_class' # 2 si sube, 1 si lateral, 0 si baja

In [17]:
from numpy.lib.stride_tricks import sliding_window_view

def create_lstm_dataset_safe(df, features, target_col, lookback=14):
    """
    Versión moderna y segura usando sliding_window_view.
    """
    # 1. Extraemos los valores
    feature_values = df[features].values
    target_values = df[target_col].values
    
    # 2. Creamos las ventanas (esto devuelve shape: [N, lookback, features])
    # sliding_window_view crea las ventanas sobre el eje 0 (filas)
    X = sliding_window_view(feature_values, window_shape=lookback, axis=0)
    
    # IMPORTANTE: sliding_window_view pone el lookback al final: (samples, features, lookback)
    # Necesitamos reordenar a (samples, lookback, features) para la LSTM
    X = X.transpose(0, 2, 1)
    
    # 3. Ajustamos el Target
    # El target 'y' debe corresponder al final de cada ventana X
    y = target_values[lookback - 1:] 
    
    # 4. Sincronización final:
    # Si y es lo que pasa DESPUÉS de la ventana, usamos:
    X = X[:-1]
    y = y[1:]
    
    return X.copy(), y.copy() # .copy() rompe el stride y evita problemas de memoria

In [23]:
class GoldLSTM_Triple_Pro(nn.Module):
    def __init__(self, input_dim, output_dim=3, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        # Usamos batch_first=True para que coincida con tu create_lstm_dataset
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        
        # Capas densas con Dropout para evitar overfitting en el Oro
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim)
        )

    def forward(self, x):
        # x shape: (batch, lookback, features)
        lstm_out, (hn, cn) = self.lstm(x)
        
        # Tomamos el último estado oculto de la última capa
        # hn[-1] shape: (batch, hidden_dim)
        return self.head(hn[-1])

In [24]:
def run_gold_master_workflow(df, feature_cols, target_col='target_class', 
                             train_size=20000, test_size=4000, step=4000, 
                             lookback=14, epochs=30, batch_size=256, lr=0.001):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    scaler_amp = amp.GradScaler('cuda')
    
    # 1. Detección de Modo
    num_classes = df[target_col].nunique()
    is_binary = (num_classes == 2)
    output_dim = 1 if is_binary else 3
    
    # 2. Pesos de Loss (Priorizamos señales 0 y 2 sobre el ruido 1)
    if is_binary:
        criterion = nn.BCEWithLogitsLoss()
        target_dtype = torch.float32
    else:
        # Pesos: [SELL=2.0, NEUTRAL=0.5, BUY=2.0] 
        # Castigamos 4 veces más el error en una señal que en el ruido
        weights = torch.tensor([2.0, 0.5, 2.0], dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=weights)
        target_dtype = torch.long

    all_predictions = []
    df_wf = df.copy().sort_values('datetime').reset_index(drop=True)
    
    # Suffix para retornos reales (extraído de tus columnas)
    suffix = "120m" # Ajustar según tu df.columns

    for start in tqdm(range(0, len(df_wf) - train_size - test_size, step), desc="Gold Master WF"):
        end_train = start + train_size
        end_test = end_train + test_size
        
        train_df = df_wf.iloc[start:end_train].copy()
        test_df = df_wf.iloc[end_train:end_test].copy()
        
        # Escalado Robusto
        sc = RobustScaler()
        train_df[feature_cols] = sc.fit_transform(train_df[feature_cols])
        test_df[feature_cols] = sc.transform(test_df[feature_cols])
        
        # Crear Datasets (Usando tu función de strides)
        X_train, y_train = create_lstm_dataset_safe(train_df, feature_cols, target_col, lookback)
        X_test, y_test = create_lstm_dataset_safe(test_df, feature_cols, target_col, lookback)
        
        # Convertir a Tensores (IMPORTANTE: .copy() para evitar problemas de strides en GPU)
        X_train_t = torch.tensor(X_train.copy(), dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=target_dtype).to(device)
        X_test_t = torch.tensor(X_test.copy(), dtype=torch.float32).to(device)
        
        if is_binary: y_train_t = y_train_t.unsqueeze(1)
        
        loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
        
        # Inicializar Modelo, Optimizer y Scheduler
        model = GoldLSTM_Triple_Pro(input_dim=len(feature_cols), output_dim=output_dim).to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
        
        # --- Bucle de Entrenamiento con Early Stopping ---
        best_loss = float('inf')
        patience_counter = 0
        early_stop_patience = 5
        
        model.train()
        for epoch in range(epochs):
            epoch_loss = 0
            for bx, by in loader:
                optimizer.zero_grad()
                with amp.autocast('cuda'):
                    outputs = model(bx)
                    loss = criterion(outputs, by)
                
                scaler_amp.scale(loss).backward()
                scaler_amp.step(optimizer)
                scaler_amp.update()
                epoch_loss += loss.item()
            
            avg_loss = epoch_loss / len(loader)
            scheduler.step(avg_loss) # El scheduler baja el LR si el loss se estanca
            
            # Lógica de Early Stopping
            if avg_loss < best_loss:
                best_loss = avg_loss
                patience_counter = 0
                # Guardar el "mejor" estado del modelo en este bloque si quisieras
                # best_model_state = model.state_dict()
            else:
                patience_counter += 1
                
            if patience_counter >= early_stop_patience:
                break # Detener entrenamiento si no mejora
        
        # --- Predicción en el bloque de Test ---
        model.eval()
        with torch.no_grad():
            logits = model(X_test_t)
            if is_binary:
                probs = torch.sigmoid(logits).cpu().numpy().ravel()
                preds = (probs > 0.5).astype(int)
                res_dict = {'prob_up': probs}
            else:
                probs = torch.softmax(logits, dim=1).cpu().numpy()
                preds = np.argmax(probs, axis=1)
                res_dict = {'prob_0': probs[:,0], 'prob_1': probs[:,1], 'prob_2': probs[:,2]}
        
        # --- Sincronización de Datetimes para el reporte ---
        # El primer target disponible en test_df empieza tras el lookback
        test_dates = test_df['datetime'].iloc[lookback:].values
        test_rets = test_df[f'target_ret_{suffix}'].iloc[lookback:].values
        
        res_df = pd.DataFrame({
            'datetime': test_dates,
            'actual': y_test, 
            'pred': preds, 
            **res_dict,
            'ret_real': test_rets
        })
        all_predictions.append(res_df)
        
        # Limpieza de memoria para el siguiente bloque
        torch.cuda.empty_cache()

    final_results = pd.concat(all_predictions, ignore_index=True)
    return final_results, model, sc

In [27]:
run_gold_master_workflow(df_final_triple_liquid, feature_cols, target_col=TARGET_COLUMN_TRIPLE)

Gold Master WF:   0%|          | 0/14 [00:07<?, ?it/s]


KeyError: 'target_ret_120m'